# Classificação de arcos e componentes — Tutorial

**Algoritmos e Estruturas de Dados 2 — DCOMP/UFS**
Prof. Dr. André Yoshiaki Kashiwabara

Continuação do tutorial de busca em profundidade. Aqui acrescentamos uma linha à
`dfsR()` — o vetor `post[]` — e colhemos três resultados: a **classificação dos
arcos**, a **detecção de ciclos** e os **componentes** — conexos e fortemente conexos.

O material segue [*Algoritmos para Grafos (em linguagem C)*](https://www.ime.usp.br/~pf/algoritmos_para_grafos/),
de Paulo Feofiloff. Todo o código é em C; as células Python apenas compilam e
executam os programas.

## Objetivos

Ao final deste tutorial você será capaz de:

- calcular `pre[]` e `post[]` e ler neles a relação de ancestralidade;
- classificar cada arco de um digrafo em árvore, retorno, avanço ou cruzado;
- decidir se um digrafo tem ciclo — e explicar por que testar "já visitado" está errado;
- verificar experimentalmente que um grafo não dirigido só tem arestas de árvore e de retorno;
- rotular os componentes conexos com `cc[]` e responder conexidade em tempo constante.
- calcular os componentes **fortemente** conexos de um digrafo com o algoritmo de Kosaraju.

O grafo condutor é o mesmo das aulas anteriores:
`0-1  0-5  1-0  1-5  2-4  3-1  5-3`.


In [ ]:
# Preparação do ambiente: cria a pasta de trabalho e confere o compilador.
import os, subprocess

os.makedirs('src', exist_ok=True)
print(subprocess.run(['gcc', '--version'], capture_output=True, text=True).stdout.splitlines()[0])

def compilar_e_rodar(fonte, entrada=None):
    """Compila src/<fonte> com gcc (C99) e executa, mostrando a saída."""
    exe = os.path.join('src', os.path.splitext(fonte)[0])
    c = subprocess.run(['gcc', '-std=c99', '-Wall', os.path.join('src', fonte), '-o', exe],
                       capture_output=True, text=True)
    if c.returncode != 0:
        print('ERRO DE COMPILACAO:\n' + c.stderr)
        return
    if c.stderr:
        print('avisos do compilador:\n' + c.stderr)
    r = subprocess.run([exe], capture_output=True, text=True, input=entrada)
    print(r.stdout, end='')
    if r.stderr:
        print('stderr:', r.stderr)


## 1. O segundo carimbo: `post[]`

`pre[v]` é gravado na **entrada** da chamada `dfsR(v)`; `post[v]`, na **saída**.
Entre os dois instantes o vértice está na pilha de recursão — dizemos que está
**aberto**.

O programa abaixo imprime cada abertura e cada fechamento. Repare que a saída é
uma sequência de parênteses balanceados: é a pilha de recursão aparecendo.


In [ ]:
%%writefile src/postorder.c
/* DFS com os DOIS carimbos: pre[] (descoberta) e post[] (termino).
   Adaptado de P. Feofiloff, "Algoritmos para Grafos (em linguagem C)". */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], pai[maxV];

static void dfsR( Graph G, vertex v) {
   printf( "   abre  %d   (pre  = %d)\n", v, cnt);
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) {
         pai[w] = v;
         dfsR( G, w);
      }
   printf( "   fecha %d   (post = %d)\n", v, cnt1);
   post[v] = cnt1++;
}

void GRAPHdfs( Graph G) {
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) { pre[v] = post[v] = -1; pai[v] = -1; }
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) {
         printf( "nova arvore, raiz %d:\n", v);
         pai[v] = v;
         dfsR( G, v);
      }
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);

   GRAPHdfs( G);

   printf( "\nv      ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", v);
   printf( "\npre    ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", pre[v]);
   printf( "\npost   ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", post[v]);
   printf( "\npai    ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", pai[v]);
   printf( "\n\npos-ordem (ordem de termino):");
   for (int k = 0; k < G->V; ++k)
      for (vertex v = 0; v < G->V; ++v)
         if (post[v] == k) printf( " %d", v);
   printf( "\n");
   return 0;
}


In [ ]:
compilar_e_rodar('postorder.c')
# Saida esperada: pre  = 0 1 4 3 5 2
#                 post = 3 2 5 0 4 1
# Pos-ordem: 3 5 1 0 4 2 -- as folhas terminam primeiro.


**Leia a saída como intervalos.** O vértice 0 abre no instante 1 e fecha no
instante 8; o 1 abre em 2 e fecha em 7; o 5, em 3 e 6; o 3, em 4 e 5. Os
intervalos estão **encaixados** — e é exatamente isso que o teorema dos
parênteses afirma:

> $w$ é descendente de $v$ na floresta DFS $\iff$ `pre[v] < pre[w]` **e** `post[v] > post[w]`.

Teste à mão com a tabela: 3 é descendente de 0? `pre[0]=0 < pre[3]=3` e
`post[0]=3 > post[3]=0`. Sim. E 2 é descendente de 0? `pre[0]=0 < pre[2]=4`,
mas `post[0]=3` não é maior que `post[2]=5`. Não.


## 2. Classificando os arcos

Com `pre[]`, `post[]` e `pai[]` em mãos, o tipo de cada arco sai de três testes.

Ao grafo da aula acrescentamos o arco `2-0`. Ele **não muda** `pre[]` nem
`post[]` (quando a busca chega a 2, o vértice 0 já fechou), mas produz o quarto
tipo de arco, que faltava: o cruzado.


In [ ]:
%%writefile src/classifica.c
/* Classificacao dos arcos de um digrafo a partir de pre[], post[] e pai[]. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], pai[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) {
         pai[w] = v;
         dfsR( G, w);
      }
   post[v] = cnt1++;
}
void GRAPHdfs( Graph G) {
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) { pre[v] = post[v] = -1; pai[v] = -1; }
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) { pai[v] = v; dfsR( G, v); }
}

/* O criterio da aula, aplicado ao arco v-w. */
static const char *tipo( vertex v, vertex w) {
   if (pre[v] < pre[w] && post[v] > post[w])       /* w descende de v */
      return (pai[w] == v) ? "arvore" : "avanco";
   if (pre[w] < pre[v] && post[w] > post[v])       /* w ancestral de v */
      return "retorno";
   return "cruzado";                               /* nenhum dos dois */
}

void GRAPHclassify( Graph G) {
   printf( "arco   pre[v] post[v] pre[w] post[w]   tipo\n");
   for (vertex v = 0; v < G->V; ++v)
      for (vertex w = 0; w < G->V; ++w)
         if (G->adj[v][w] != 0)
            printf( "%d-%d %8d %6d %7d %6d   %s\n",
                    v, w, pre[v], post[v], pre[w], post[w], tipo( v, w));
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);
   GRAPHinsertArc( G, 2, 0);    /* o arco extra que produz o arco cruzado */

   GRAPHdfs( G);
   GRAPHclassify( G);
   return 0;
}


In [ ]:
compilar_e_rodar('classifica.c')
# Confira contra a tabela dos oito arcos da aula:
#   arvore : 0-1, 1-5, 5-3, 2-4
#   avanco : 0-5      (5 descende de 0, mas pai[5] = 1)
#   retorno: 1-0, 3-1
#   cruzado: 2-0


### Exercício 1 — classificar **durante** a busca

O programa acima espera a busca terminar para depois classificar. Dá para
classificar cada arco no instante em que ele é examinado, dentro de `dfsR()` —
é assim que a classificação costuma aparecer na prática, porque não exige um
segundo passeio pelo grafo.

A dificuldade é que, naquele momento, `post[]` ainda está incompleto. Em
compensação, ele carrega a informação de que você precisa: cada vértice está em
um de três estados, e os dois vetores juntos os distinguem.

Complete `tipoDurante()` no esqueleto abaixo. O gabarito é a tabela dos oito
arcos.


In [ ]:
%%writefile src/ex1_durante.c
/* Exercicio 1: classificar cada arco DURANTE a busca, sem esperar o fim. */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV];
static char tipoDe[maxV][maxV][8];    /* tipoDe[v][w] = tipo do arco v-w */

/* TODO: devolva o tipo do arco v-w no instante em que dfsR(v) o examina.
   Nesse instante voce NAO tem post[] completo -- tem apenas tres estados:
     w branco  (nunca descoberto), w cinza (aberto), w preto (ja fechado).
   Descubra como ler esses tres estados em pre[w] e post[w], e como separar
   "avanco" de "cruzado" entre os pretos comparando pre[v] com pre[w].
   Devolva uma das strings: "arvore", "retorno", "avanco", "cruzado". */
static const char *tipoDurante( vertex v, vertex w) {
   (void) v; (void) w;
   return "?";                        /* implemente aqui */
}

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0) {
         strcpy( tipoDe[v][w], tipoDurante( v, w));   /* classifica na hora */
         if (pre[w] == -1) dfsR( G, w);
      }
   post[v] = cnt1++;
}
static void GRAPHdfs( Graph G) {
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) pre[v] = post[v] = -1;
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) dfsR( G, v);
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3); GRAPHinsertArc( G, 2, 0);

   GRAPHdfs( G);

   /* Gabarito: a tabela dos oito arcos da aula. */
   vertex ev[] = {0,0,1,1,2,2,3,5};
   vertex ew[] = {1,5,0,5,0,4,1,3};
   const char *et[] = {"arvore","avanco","retorno","arvore",
                       "cruzado","arvore","retorno","arvore"};
   int ok = 1;
   for (int i = 0; i < 8; ++i) {
      printf( "%d-%d  seu: %-8s  esperado: %s\n",
              ev[i], ew[i], tipoDe[ev[i]][ew[i]], et[i]);
      if (strcmp( tipoDe[ev[i]][ew[i]], et[i]) != 0) ok = 0;
   }
   printf( "\n%s\n", ok ? "OK: todos os oito arcos classificados corretamente."
                        : "FALHOU: ainda ha arcos com tipo errado.");
   return 0;
}


In [ ]:
compilar_e_rodar('ex1_durante.c')


## 3. Ciclos e arcos de retorno

> Um digrafo tem ciclo **se e somente se** a busca em profundidade produz pelo
> menos um arco de retorno.

E "arco de retorno" quer dizer que o destino está **aberto** (ainda na pilha),
não que ele "já foi visto". A diferença não é cosmética: o programa abaixo roda
a versão certa e a errada lado a lado, e mostra a versão errada acusando ciclo
em um grafo que não tem nenhum.


In [ ]:
%%writefile src/ciclo.c
/* Deteccao de ciclo em digrafo: a versao certa e a versao errada, lado a lado. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV];
static int temCiclo;

/* CERTA: so o vertice ainda ABERTO (cinza) acusa arco de retorno. */
static void dfsCerta( Graph G, vertex v) {
   pre[v] = cnt++;                                 /* v fica CINZA */
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0) {
         if (pre[w] == -1) dfsCerta( G, w);              /* branco */
         else if (post[w] == -1) {                       /* CINZA  */
            if (!temCiclo) printf( "   arco de retorno: %d-%d\n", v, w);
            temCiclo = 1;
         }
      }                                                  /* preto: ignore */
   post[v] = cnt1++;                               /* v fica PRETO */
}

/* ERRADA: acusa qualquer vertice ja visto, inclusive avanco e cruzado. */
static void dfsErrada( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0) {
         if (pre[w] == -1) dfsErrada( G, w);
         else {
            if (!temCiclo) printf( "   acusou o arco: %d-%d\n", v, w);
            temCiclo = 1;
         }
      }
   post[v] = cnt1++;
}

static int roda( Graph G, int errada) {
   cnt = cnt1 = 0; temCiclo = 0;
   for (vertex v = 0; v < G->V; ++v) pre[v] = post[v] = -1;
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) {
         if (errada) dfsErrada( G, v); else dfsCerta( G, v);
      }
   return temCiclo;
}

static void testa( const char *nome, Graph G) {
   printf( "%s\n", nome);
   printf( "  versao CERTA :\n");
   printf( "   -> %s\n", roda( G, 0) ? "TEM ciclo" : "NAO tem ciclo");
   printf( "  versao ERRADA:\n");
   printf( "   -> %s\n\n", roda( G, 1) ? "TEM ciclo" : "NAO tem ciclo");
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 4); GRAPHinsertArc( G, 3, 1);
   GRAPHinsertArc( G, 5, 3);
   testa( "grafo da aula (tem ciclo: 0-1-0 e 1-5-3-1)", G);

   Graph D = GRAPHinit( 3);          /* um DAG: 0->1->2 e o atalho 0->2 */
   GRAPHinsertArc( D, 0, 1); GRAPHinsertArc( D, 1, 2); GRAPHinsertArc( D, 0, 2);
   testa( "DAG 0-1, 1-2, 0-2 (o arco 0-2 e de AVANCO, nao de retorno)", D);
   return 0;
}


In [ ]:
compilar_e_rodar('ciclo.c')
# No DAG 0->1->2 com o atalho 0->2, o arco 0-2 e de AVANCO.
# A versao errada o confunde com um arco de retorno e acusa um ciclo inexistente.


### Exercício 2 — `GRAPHcyclic` com três cores

Escreva a detecção de ciclo usando um vetor `cor[]` explícito, com os valores
`BRANCO`, `CINZA` e `PRETO`, em vez de deduzir o estado de `pre[]` e `post[]`.
A versão com cores é a de Cormen *et al.*; a com `pre`/`post` é a de Feofiloff.
São a mesma coisa, e é bom enxergar isso.

Os quatro casos de teste incluem um DAG com arco de avanço e um DAG com arco
cruzado — os dois que a implementação ingênua erra.


In [ ]:
%%writefile src/ex2_ciclo.c
/* Exercicio 2: GRAPHcyclic -- o digrafo tem ciclo? */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

#define BRANCO 0
#define CINZA  1
#define PRETO  2
static int cor[maxV];
static int temCiclo;

/* TODO: percorra v marcando-o de CINZA na entrada e de PRETO na saida.
   Um arco v-w com w CINZA e um arco de retorno: faca temCiclo = 1.
   Cuidado: w PRETO nao fecha ciclo nenhum. */
static void dfsRciclo( Graph G, vertex v) {
   (void) G; (void) v;
   /* implemente aqui */
}

int GRAPHcyclic( Graph G) {
   temCiclo = 0;
   for (vertex v = 0; v < G->V; ++v) cor[v] = BRANCO;
   for (vertex v = 0; v < G->V; ++v)
      if (cor[v] == BRANCO) dfsRciclo( G, v);
   return temCiclo;
}

int main( void) {
   int ok = 1;

   Graph A = GRAPHinit( 6);          /* grafo da aula: TEM ciclo */
   GRAPHinsertArc( A, 0, 1); GRAPHinsertArc( A, 0, 5);
   GRAPHinsertArc( A, 1, 0); GRAPHinsertArc( A, 1, 5);
   GRAPHinsertArc( A, 2, 4); GRAPHinsertArc( A, 3, 1);
   GRAPHinsertArc( A, 5, 3);

   Graph B = GRAPHinit( 3);          /* DAG com arco de avanco: NAO tem */
   GRAPHinsertArc( B, 0, 1); GRAPHinsertArc( B, 1, 2); GRAPHinsertArc( B, 0, 2);

   Graph C = GRAPHinit( 4);          /* dois ramos + arco cruzado: NAO tem */
   GRAPHinsertArc( C, 0, 1); GRAPHinsertArc( C, 0, 2); GRAPHinsertArc( C, 2, 1);

   Graph D = GRAPHinit( 3);          /* triangulo dirigido: TEM ciclo */
   GRAPHinsertArc( D, 0, 1); GRAPHinsertArc( D, 1, 2); GRAPHinsertArc( D, 2, 0);

   Graph *g[] = {&A, &B, &C, &D};
   const char *nome[] = {"grafo da aula", "DAG com avanco",
                         "DAG com cruzado", "triangulo dirigido"};
   int esperado[] = {1, 0, 0, 1};
   for (int i = 0; i < 4; ++i) {
      int r = GRAPHcyclic( *g[i]);
      printf( "%-20s  seu: %d  esperado: %d\n", nome[i], r, esperado[i]);
      if (r != esperado[i]) ok = 0;
   }
   printf( "\n%s\n", ok ? "OK: GRAPHcyclic acertou os quatro casos."
                        : "FALHOU: reveja quando um arco e de RETORNO.");
   return 0;
}


In [ ]:
compilar_e_rodar('ex2_ciclo.c')


## 4. Grafos não dirigidos: só árvore e retorno

Em um grafo não dirigido não existem arestas de avanço nem cruzadas. O
argumento da aula: se ao examinar $\{v,w\}$ o vértice $w$ já estivesse fechado
sem ser ancestral de $v$, então, quando a busca estava em $w$, ela teria
examinado $\{w,v\}$ com $v$ ainda branco — e $v$ teria virado descendente de
$w$.

Duas observações sobre o código:

- cada aresta é examinada **duas vezes**, uma de cada ponta; classificamos na
  primeira e ignoramos na segunda;
- o teste `w != p` evita confundir a aresta de onde viemos com um retorno;
- inserimos os vizinhos no **fim** da lista (e não no início) só para que saiam
  em ordem crescente e o resultado bata com o da aula.


In [ ]:
%%writefile src/naodirigido.c
/* Grafo NAO DIRIGIDO: a DFS so produz arestas de arvore e de retorno. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
/* Insere no FIM da lista: assim os vizinhos saem na ordem em que
   foram inseridos (aqui, crescente) e o resultado bate com o da aula. */
static void insereFim( Graph G, vertex v, vertex w) {
   link novo = malloc( sizeof *novo);
   novo->w = w; novo->next = NULL;
   if (G->adj[v] == NULL) { G->adj[v] = novo; return; }
   link a = G->adj[v];
   while (a->next != NULL) a = a->next;
   a->next = novo;
}
void GRAPHinsertEdge( Graph G, vertex v, vertex w) {
   insereFim( G, v, w);
   insereFim( G, w, v);      /* uma aresta = duas entradas */
   G->A++;
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], pai[maxV];
static int nArvore, nRetorno, nRevisita, nAnomalia;

static void dfsND( Graph G, vertex v, vertex p) {
   pre[v] = cnt++;
   for (link a = G->adj[v]; a != NULL; a = a->next) {
      vertex w = a->w;
      if (pre[w] == -1) {                    /* w branco: avanco a busca */
         pai[w] = v; nArvore++;
         printf( "   %d-%d  arvore\n", v, w);
         dfsND( G, w, v);
      }
      else if (post[w] == -1) {              /* w ainda ABERTO */
         if (w != p) {                       /* nao e a aresta por onde vim */
            nRetorno++;
            printf( "   %d-%d  RETORNO  (fecha um ciclo)\n", v, w);
         }
      }
      else {
         /* w ja FECHOU. Como v ainda esta aberto, w so pode ter fechado
            dentro da chamada dfsND(v) -- ou seja, w e descendente de v e
            esta aresta ja foi classificada como RETORNO pela ponta w.
            Se um dia pre[w] < pre[v] aqui, havera aresta cruzada; a aula
            prova que isso nao acontece. */
         nRevisita++;
         if (pre[w] < pre[v]) nAnomalia++;
      }
   }
   post[v] = cnt1++;
}

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertEdge( G, 0, 1); GRAPHinsertEdge( G, 0, 5);
   GRAPHinsertEdge( G, 1, 3); GRAPHinsertEdge( G, 1, 5);
   GRAPHinsertEdge( G, 2, 4); GRAPHinsertEdge( G, 3, 5);

   cnt = cnt1 = 0; nArvore = nRetorno = nRevisita = nAnomalia = 0;
   for (vertex v = 0; v < G->V; ++v) { pre[v] = post[v] = -1; pai[v] = -1; }
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) {
         printf( "nova arvore, raiz %d:\n", v);
         pai[v] = v;
         dfsND( G, v, v);
      }

   printf( "\nv      ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", v);
   printf( "\npre    ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", pre[v]);
   printf( "\npost   ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", post[v]);
   printf( "\n\narestas de arvore       : %d", nArvore);
   printf( "\narestas de retorno      : %d", nRetorno);
   printf( "\narvore + retorno        : %d   (total de arestas: %d)",
           nArvore + nRetorno, G->A);
   printf( "\nrevistas pela 2a ponta  : %d   (ja tinham tipo)", nRevisita);
   printf( "\navanco ou cruzada       : %d   <-- sempre 0\n", nAnomalia);
   return 0;
}


In [ ]:
compilar_e_rodar('naodirigido.c')
# Saida esperada: arvore {0,1} {1,3} {3,5} {2,4}; retorno {5,0} {5,1};
# e o contador de "avanco ou cruzada" em ZERO.


## 5. Componentes conexos

Em um grafo não dirigido, uma única busca a partir de $s$ descobre exatamente o
componente de $s$. Logo a varredura descobre **um componente por árvore** da
floresta, e basta rotular: `cc[v]` é o número do componente de `v`.

Repare que `cc[]` substitui `pre[]` como marcador — um vetor só faz os dois
papéis, como `pre[]` fazia na aula passada.


In [ ]:
%%writefile src/componentes.c
/* Componentes conexos de um grafo NAO DIRIGIDO: o vetor cc[]. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
static void insereFim( Graph G, vertex v, vertex w) {
   link novo = malloc( sizeof *novo);
   novo->w = w; novo->next = NULL;
   if (G->adj[v] == NULL) { G->adj[v] = novo; return; }
   link a = G->adj[v];
   while (a->next != NULL) a = a->next;
   a->next = novo;
}
void GRAPHinsertEdge( Graph G, vertex v, vertex w) {
   insereFim( G, v, w); insereFim( G, w, v); G->A++;
}

static int cc[maxV];    /* cc[v] = numero do componente de v; -1 = sem rotulo */

static void dfsRcc( Graph G, vertex v, int c) {
   cc[v] = c;                                  /* cc[] tambem MARCA o vertice */
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (cc[a->w] == -1)
         dfsRcc( G, a->w, c);
}

int GRAPHcc( Graph G) {
   int c = 0;
   for (vertex v = 0; v < G->V; ++v) cc[v] = -1;
   for (vertex v = 0; v < G->V; ++v)
      if (cc[v] == -1) dfsRcc( G, v, c++);
   return c;                                   /* numero de componentes */
}

/* Depois do pre-processamento, cada consulta custa tempo CONSTANTE. */
int GRAPHconnected( vertex v, vertex w) { return cc[v] == cc[w]; }

int main( void) {
   Graph G = GRAPHinit( 6);
   GRAPHinsertEdge( G, 0, 1); GRAPHinsertEdge( G, 0, 5);
   GRAPHinsertEdge( G, 1, 3); GRAPHinsertEdge( G, 1, 5);
   GRAPHinsertEdge( G, 2, 4); GRAPHinsertEdge( G, 3, 5);

   int k = GRAPHcc( G);
   printf( "numero de componentes: %d\n\n", k);
   printf( "v      ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", v);
   printf( "\ncc[v]  ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", cc[v]);
   printf( "\n\n");
   for (int c = 0; c < k; ++c) {
      printf( "componente %d: {", c);
      for (vertex v = 0; v < G->V; ++v) if (cc[v] == c) printf( " %d", v);
      printf( " }\n");
   }
   printf( "\nconsultas em tempo constante:\n");
   printf( "   0 e 3 conectados? %s\n", GRAPHconnected( 0, 3) ? "sim" : "nao");
   printf( "   0 e 4 conectados? %s\n", GRAPHconnected( 0, 4) ? "sim" : "nao");
   printf( "   2 e 4 conectados? %s\n", GRAPHconnected( 2, 4) ? "sim" : "nao");
   return 0;
}


In [ ]:
compilar_e_rodar('componentes.c')
# Dois componentes: {0,1,3,5} e {2,4}.


**O padrão que se repete no curso:** gaste $\Theta(V+A)$ *uma vez* no
pré-processamento e responda a cada consulta "$v$ e $w$ estão conectados?" em
tempo **constante**, com `cc[v] == cc[w]`. Sem isso, cada pergunta custaria uma
busca inteira.

### Exercício 3 — componentes do Exemplo A

Implemente `GRAPHcc()`, `dfsRcc()` e `GRAPHmaiorCC()` para a versão **não
dirigida** do Exemplo A (o grafo de 12 vértices do tutorial anterior). Um dos
vértices fica isolado — encontre qual.


In [ ]:
%%writefile src/ex3_cc.c
/* Exercicio 3: componentes conexos e o tamanho do maior deles. */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertEdge( Graph G, vertex v, vertex w) {   /* NAO dirigido */
   if (G->adj[v][w] == 0) { G->adj[v][w] = G->adj[w][v] = 1; G->A++; }
}

static int cc[maxV];

/* TODO: rotule com c todos os vertices alcancaveis a partir de v.
   Lembre: cc[] tambem faz o papel de marcador (-1 = ainda sem rotulo). */
void dfsRcc( Graph G, vertex v, int c) {
   (void) G; (void) v; (void) c;
   /* implemente aqui */
}

/* TODO: rotule todos os vertices e devolva o NUMERO de componentes. */
int GRAPHcc( Graph G) {
   for (vertex v = 0; v < G->V; ++v) cc[v] = -1;
   /* implemente aqui */
   return -1;
}

/* TODO: devolva o numero de vertices do MAIOR componente.
   Suponha que GRAPHcc(G) ja foi chamada. */
int GRAPHmaiorCC( Graph G, int k) {
   (void) G; (void) k;
   return -1;                         /* implemente aqui */
}

int GRAPHconnected( vertex v, vertex w) { return cc[v] == cc[w]; }

int main( void) {
   /* Versao nao dirigida do Exemplo A (12 vertices; o 9 fica isolado). */
   int e[][2] = {{0,5},{0,6},{2,0},{2,3},{3,6},{3,10},{4,1},{5,2},
                 {5,10},{6,2},{7,8},{7,11},{8,1},{8,4},{10,3},{11,8}};
   Graph G = GRAPHinit( 12);
   for (int i = 0; i < 16; ++i) GRAPHinsertEdge( G, e[i][0], e[i][1]);

   int k = GRAPHcc( G);
   int maior = GRAPHmaiorCC( G, k);
   printf( "numero de componentes : seu: %2d  esperado:  3\n", k);
   printf( "maior componente      : seu: %2d  esperado:  6\n", maior);
   printf( "0 e 10 conectados?    : seu: %2d  esperado:  1\n", GRAPHconnected( 0, 10));
   printf( "0 e  9 conectados?    : seu: %2d  esperado:  0\n", GRAPHconnected( 0, 9));
   int ok = (k == 3 && maior == 6 &&
             GRAPHconnected( 0, 10) == 1 && GRAPHconnected( 0, 9) == 0);
   printf( "\n%s\n", ok ? "OK: componentes corretos."
                        : "FALHOU: reveja GRAPHcc e GRAPHmaiorCC.");
   return 0;
}


In [ ]:
compilar_e_rodar('ex3_cc.c')


## 6. Componentes fortemente conexos: o algoritmo de Kosaraju

Em um digrafo, "estar no mesmo componente" só faz sentido se a ida **e** a volta
existirem.

> **Definição.** Em um digrafo, $v \equiv w$ quando existe caminho de $v$ para
> $w$ **e** caminho de $w$ para $v$. Um **componente fortemente conexo** é um
> conjunto *máximo* de vértices dois a dois equivalentes; o digrafo é
> **fortemente conexo** quando tem um componente só.

A relação $\equiv$ é de equivalência — reflexiva (caminho de comprimento zero),
simétrica (a definição exige os dois sentidos) e transitiva (concatene os
caminhos) —, então os componentes **particionam** os vértices. A floresta da DFS
não responde a essa pergunta: o número de árvores depende da raiz escolhida.

O algoritmo de Kosaraju resolve em $\Theta(V+A)$, com duas buscas:

1. rode a DFS em $G$ e guarde `post[]`;
2. construa $G^R$, o mesmo grafo com **todos os arcos invertidos**;
3. rode a varredura em $G^R$, escolhendo as raízes em ordem **decrescente** de
   `post`. Cada árvore dessa segunda floresta é um componente fortemente conexo.

Por que funciona: se há arco de um componente $C$ para outro $C'$, então o maior
`post` de $C$ é maior que o maior `post` de $C'$. Logo o vértice de `post` máximo
está em um componente *fonte* — e em $G^R$ esse componente vira *sumidouro*, do
qual a busca não consegue escapar.

In [ ]:
%%writefile src/kosaraju.c
/* Componentes fortemente conexos: o algoritmo de Kosaraju (duas buscas). */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

typedef struct node *link;
struct node { vertex w; link next; };
struct graph { int V; int A; link *adj; };
typedef struct graph *Graph;

Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0;
   G->adj = malloc( V * sizeof (link));
   for (vertex v = 0; v < V; ++v) G->adj[v] = NULL;
   return G;
}
static void insereFim( Graph G, vertex v, vertex w) {
   link novo = malloc( sizeof *novo);
   novo->w = w; novo->next = NULL;
   if (G->adj[v] == NULL) { G->adj[v] = novo; return; }
   link a = G->adj[v];
   while (a->next != NULL) a = a->next;
   a->next = novo;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {   /* DIRIGIDO */
   insereFim( G, v, w); G->A++;
}

/* Passo 2: o grafo reverso. Cada arco v-w de G vira w-v em GR. */
Graph GRAPHreverse( Graph G) {
   Graph GR = GRAPHinit( G->V);
   for (vertex v = 0; v < G->V; ++v)
      for (link a = G->adj[v]; a != NULL; a = a->next)
         GRAPHinsertArc( GR, a->w, v);
   return GR;
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], ord[maxV], sc[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (link a = G->adj[v]; a != NULL; a = a->next)
      if (pre[a->w] == -1) dfsR( G, a->w);
   post[v] = cnt1++;
}
void GRAPHdfs( Graph G) {                 /* passo 1: preenche post[] */
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) dfsR( G, v);
}

/* A mesma dfsRcc() da secao anterior, agora rodando em GR e rotulando sc[]. */
static void dfsRsc( Graph GR, vertex v, int c) {
   sc[v] = c;
   for (link a = GR->adj[v]; a != NULL; a = a->next)
      if (sc[a->w] == -1) dfsRsc( GR, a->w, c);
}

int GRAPHscc( Graph G) {
   GRAPHdfs( G);                          /* 1: post[]                 */
   Graph GR = GRAPHreverse( G);           /* 2: arcos invertidos       */
   for (vertex v = 0; v < G->V; ++v) {    /* ord = post DECRESCENTE    */
      ord[G->V-1 - post[v]] = v;          /* post[] e uma permutacao:  */
      sc[v] = -1;                         /* ordenar custa Theta(V)    */
   }
   int c = 0;
   for (int i = 0; i < G->V; ++i)         /* 3: raizes na ordem de ord */
      if (sc[ord[i]] == -1) dfsRsc( GR, ord[i], c++);
   return c;
}

int main( void) {
   Graph G = GRAPHinit( 6);               /* o digrafo da aula */
   GRAPHinsertArc( G, 0, 1); GRAPHinsertArc( G, 0, 5);
   GRAPHinsertArc( G, 1, 0); GRAPHinsertArc( G, 1, 5);
   GRAPHinsertArc( G, 2, 0); GRAPHinsertArc( G, 2, 4);
   GRAPHinsertArc( G, 3, 1); GRAPHinsertArc( G, 5, 3);

   int k = GRAPHscc( G);

   printf( "v       ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", v);
   printf( "\npost[v] ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", post[v]);
   printf( "\nsc[v]   ");
   for (vertex v = 0; v < G->V; ++v) printf( "%3d", sc[v]);
   printf( "\n\nraizes da 2a busca, em post decrescente:");
   for (int i = 0; i < G->V; ++i) printf( " %d", ord[i]);
   printf( "\n\ncomponentes fortemente conexos: %d\n", k);
   for (int c = 0; c < k; ++c) {
      printf( "   componente %d: {", c);
      for (vertex v = 0; v < G->V; ++v) if (sc[v] == c) printf( " %d", v);
      printf( " }\n");
   }
   printf( "\n0 e 3 sao equivalentes? %s\n", sc[0] == sc[3] ? "sim" : "nao");
   printf( "0 e 2 sao equivalentes? %s\n", sc[0] == sc[2] ? "sim" : "nao");
   return 0;
}


In [ ]:
compilar_e_rodar('kosaraju.c')
# Saida esperada: sc[] = 2 2 0 2 1 2, com as raizes da 2a busca em 2 4 0 1 5 3
# e tres componentes: {2}, {4} e {0,1,3,5}.

Compare com a seção 5: no grafo **não dirigido** havia dois componentes
(`{0,1,3,5}` e `{2,4}`); no **digrafo** são três, porque 2 alcança 4 mas 4 não
alcança 2. Repare também na ordem dos rótulos: eles saem em ordem *topológica*
da condensação — o componente `{2}`, que é a fonte, recebe o rótulo 0.

Experimente trocar o passo 3 por `post` **crescente** e veja o resultado
desmoronar: a segunda busca começaria por um sumidouro de $G$, que em $G^R$ vira
fonte e alcança o grafo inteiro.

### Exercício 4 — Kosaraju no Exemplo A

Implemente `GRAPHreverse()`, `dfsRsc()`, `GRAPHscc()` e `GRAPHmaiorSCC()` para a
versão **dirigida** do Exemplo A. O teste confere o número de componentes fortes
e o tamanho do maior. Guarde o contraste: a versão não dirigida do mesmo grafo
tem 3 componentes; a dirigida tem bem mais.

In [ ]:
%%writefile src/ex4_kosaraju.c
/* Exercicio 4: componentes fortemente conexos do Exemplo A (Kosaraju). */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {   /* DIRIGIDO */
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], ord[maxV], sc[maxV];

static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) dfsR( G, w);
   post[v] = cnt1++;
}
void GRAPHdfs( Graph G) {
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) pre[v] = -1;
   for (vertex v = 0; v < G->V; ++v) if (pre[v] == -1) dfsR( G, v);
}

/* TODO: devolva o grafo com TODOS os arcos invertidos (a transposta). */
Graph GRAPHreverse( Graph G) {
   Graph GR = GRAPHinit( G->V);
   /* implemente aqui */
   return GR;
}

/* TODO: rotule com c tudo que e alcancavel a partir de v EM GR.
   E a mesma dfsRcc() do exercicio 3, trocando cc[] por sc[]. */
static void dfsRsc( Graph GR, vertex v, int c) {
   (void) GR; (void) v; (void) c;
   /* implemente aqui */
}

/* TODO: os tres passos de Kosaraju. Devolva o NUMERO de componentes.
   1) GRAPHdfs(G) preenche post[];
   2) GR = GRAPHreverse(G);
   3) varra GR com as raizes em post DECRESCENTE (use ord[]).
   Dica: post[] e uma permutacao de 0..V-1, entao
   ord[G->V-1 - post[v]] = v ordena em Theta(V), sem qsort. */
int GRAPHscc( Graph G) {
   GRAPHdfs( G);
   for (vertex v = 0; v < G->V; ++v) sc[v] = -1;
   /* implemente aqui */
   return -1;
}

/* TODO: numero de vertices do MAIOR componente forte.
   Suponha que GRAPHscc(G) ja foi chamada. */
int GRAPHmaiorSCC( Graph G, int k) {
   (void) G; (void) k;
   return -1;                         /* implemente aqui */
}

int main( void) {
   /* Exemplo A, agora DIRIGIDO (os mesmos 16 arcos do tutorial anterior). */
   int e[][2] = {{0,5},{0,6},{2,0},{2,3},{3,6},{3,10},{4,1},{5,2},
                 {5,10},{6,2},{7,8},{7,11},{8,1},{8,4},{10,3},{11,8}};
   Graph G = GRAPHinit( 12);
   for (int i = 0; i < 16; ++i) GRAPHinsertArc( G, e[i][0], e[i][1]);

   int k = GRAPHscc( G);
   int maior = GRAPHmaiorSCC( G, k);
   printf( "componentes fortes : seu: %2d  esperado:  7\n", k);
   printf( "maior componente   : seu: %2d  esperado:  6\n", maior);
   printf( "0 e 10 equivalentes: seu: %2d  esperado:  1\n", sc[0] == sc[10]);
   printf( "7 e  8 equivalentes: seu: %2d  esperado:  0\n", sc[7] == sc[8]);
   int ok = (k == 7 && maior == 6 && sc[0] == sc[10] && sc[7] != sc[8]);
   printf( "\n%s\n", ok ? "OK: componentes fortes corretos."
                        : "FALHOU: reveja GRAPHreverse, dfsRsc e GRAPHscc.");
   return 0;
}


In [ ]:
compilar_e_rodar('ex4_kosaraju.c')

## Desafio Final — o relatório estrutural de um digrafo

Escreva um programa que leia um **arquivo de arcos** (primeira linha $V$,
segunda $A$, depois $A$ pares `v w`) e produza, em uma única varredura, o
retrato estrutural do digrafo:

1. `pre[]`, `post[]` e o número de árvores da floresta DFS;
2. a contagem de arcos por tipo, listando os que não são de árvore;
3. se o digrafo tem ciclo;
4. quantos componentes conexos tem a versão não dirigida do mesmo grafo.

O programa abaixo é uma solução completa — leia-a como referência e depois
compare os itens 1 e 4: eles **não** dão o mesmo número, e entender por quê é o
ponto da aula.


In [ ]:
# Arquivo de arcos do Exemplo A (o mesmo do tutorial anterior).
arcos = [(0,5),(0,6),(2,0),(2,3),(3,6),(3,10),(4,1),(5,2),
         (5,10),(6,2),(7,8),(7,11),(8,1),(8,4),(10,3),(11,8)]
with open('src/exemploA.txt', 'w') as f:
    f.write('12\n%d\n' % len(arcos))
    for v, w in arcos:
        f.write('%d %d\n' % (v, w))
print(open('src/exemploA.txt').read())


In [ ]:
%%writefile src/desafio.c
/* DESAFIO: le um arquivo de arcos e produz o relatorio estrutural do digrafo.
   Formato da entrada: V, depois A, depois A pares "v w". */
#include <stdio.h>
#include <stdlib.h>

#define vertex int
#define maxV 100

struct graph { int V; int A; int **adj; };
typedef struct graph *Graph;

static int **MATRIXint( int r, int c, int val) {
   int **m = malloc( r * sizeof (int *));
   for (vertex i = 0; i < r; ++i) m[i] = malloc( c * sizeof (int));
   for (vertex i = 0; i < r; ++i)
      for (vertex j = 0; j < c; ++j) m[i][j] = val;
   return m;
}
Graph GRAPHinit( int V) {
   Graph G = malloc( sizeof *G);
   G->V = V; G->A = 0; G->adj = MATRIXint( V, V, 0);
   return G;
}
void GRAPHinsertArc( Graph G, vertex v, vertex w) {
   if (G->adj[v][w] == 0) { G->adj[v][w] = 1; G->A++; }
}

static int cnt, cnt1;
static int pre[maxV], post[maxV], pai[maxV], cc[maxV];

/* ---------- parte dirigida: pre, post, pai, classificacao, ciclo ---------- */
static void dfsR( Graph G, vertex v) {
   pre[v] = cnt++;
   for (vertex w = 0; w < G->V; ++w)
      if (G->adj[v][w] != 0 && pre[w] == -1) { pai[w] = v; dfsR( G, w); }
   post[v] = cnt1++;
}
static int GRAPHdfs( Graph G) {              /* devolve o numero de arvores */
   int raizes = 0;
   cnt = cnt1 = 0;
   for (vertex v = 0; v < G->V; ++v) { pre[v] = post[v] = -1; pai[v] = -1; }
   for (vertex v = 0; v < G->V; ++v)
      if (pre[v] == -1) { pai[v] = v; raizes++; dfsR( G, v); }
   return raizes;
}
static const char *tipo( vertex v, vertex w) {
   if (pre[v] < pre[w] && post[v] > post[w])
      return (pai[w] == v) ? "arvore" : "avanco";
   if (pre[w] < pre[v] && post[w] > post[v]) return "retorno";
   return "cruzado";
}

/* ---------- parte nao dirigida: componentes conexos ---------- */
static void dfsRcc( Graph G, vertex v, int c) {
   cc[v] = c;
   for (vertex w = 0; w < G->V; ++w)
      /* versao nao dirigida: ha aresta se ha arco em qualquer sentido */
      if ((G->adj[v][w] != 0 || G->adj[w][v] != 0) && cc[w] == -1)
         dfsRcc( G, w, c);
}
static int GRAPHcc( Graph G) {
   int c = 0;
   for (vertex v = 0; v < G->V; ++v) cc[v] = -1;
   for (vertex v = 0; v < G->V; ++v)
      if (cc[v] == -1) dfsRcc( G, v, c++);
   return c;
}

int main( void) {
   int V, A;
   if (scanf( "%d %d", &V, &A) != 2) return 1;
   Graph G = GRAPHinit( V);
   for (int i = 0; i < A; ++i) {
      vertex v, w;
      if (scanf( "%d %d", &v, &w) != 2) return 1;
      GRAPHinsertArc( G, v, w);
   }
   printf( "digrafo com V = %d e A = %d\n\n", G->V, G->A);

   int raizes = GRAPHdfs( G);

   int n[4] = {0,0,0,0};       /* arvore, avanco, retorno, cruzado */
   printf( "arcos que NAO sao de arvore:\n");
   for (vertex v = 0; v < G->V; ++v)
      for (vertex w = 0; w < G->V; ++w)
         if (G->adj[v][w] != 0) {
            const char *t = tipo( v, w);
            if (t[0] == 'a' && t[1] == 'r') n[0]++;
            else if (t[0] == 'a') { n[1]++; printf( "   %2d-%-2d  avanco\n", v, w); }
            else if (t[0] == 'r') { n[2]++; printf( "   %2d-%-2d  RETORNO\n", v, w); }
            else { n[3]++; printf( "   %2d-%-2d  cruzado\n", v, w); }
         }

   printf( "\narvore  %d | avanco %d | retorno %d | cruzado %d   (soma = %d)\n",
           n[0], n[1], n[2], n[3], n[0]+n[1]+n[2]+n[3]);
   printf( "arvores na floresta DFS      : %d\n", raizes);
   printf( "o digrafo tem ciclo?         : %s   (ha %d arco(s) de retorno)\n",
           n[2] > 0 ? "SIM" : "NAO", n[2]);
   printf( "componentes conexos (versao nao dirigida): %d\n", GRAPHcc( G));
   printf( "\nv    "); for (vertex v = 0; v < V; ++v) printf( "%3d", v);
   printf( "\npre  "); for (vertex v = 0; v < V; ++v) printf( "%3d", pre[v]);
   printf( "\npost "); for (vertex v = 0; v < V; ++v) printf( "%3d", post[v]);
   printf( "\ncc   "); for (vertex v = 0; v < V; ++v) printf( "%3d", cc[v]);
   printf( "\n");
   return 0;
}


In [ ]:
compilar_e_rodar('desafio.c', entrada=open('src/exemploA.txt').read())
# Repare no contraste: 5 arvores na floresta DFS, mas apenas 3 componentes
# conexos. O numero de arvores depende da ordem das raizes; a particao em
# componentes, nao.


### Sua conclusão

Responda aqui (edite esta célula):

- Quantos arcos de cada tipo tem o Exemplo A? ____
- A floresta DFS tem 5 árvores e a versão não dirigida tem 3 componentes.
  Explique, em uma frase, por que esses números podem diferir. ____
- Rode a busca começando pelo vértice 7 em vez do 0 (basta reordenar o laço de
  `GRAPHdfs`). O número de árvores muda? E o número de componentes? ____
- O Exemplo A tem ciclo. Aponte um arco de retorno e escreva o ciclo que ele
  fecha, seguindo `pai[]`. ____
- Qual vértice fica isolado na versão não dirigida, e como isso aparece em
  `cc[]`? ____


## Referências

- FEOFILOFF, P. *Algoritmos para Grafos (em linguagem C)*. IME-USP, 2020.
  Capítulos [Busca em profundidade](https://www.ime.usp.br/~pf/algoritmos_para_grafos/aulas/dfs.html)
  e seguintes (classificação de arcos, ciclos, componentes).
- CORMEN, T. H. et al. *Introduction to Algorithms*. 3. ed. MIT Press, 2009.
  Seção 22.3 (teorema dos parênteses e classificação de arestas) e 22.5
  (componentes fortemente conexos).
- SEDGEWICK, R. *Algorithms in C, Part 5: Graph Algorithms*. 3. ed.
  Addison-Wesley, 2002. Cap. 19.
- TARJAN, R. Depth-First Search and Linear Graph Algorithms.
  *SIAM J. Comput.*, v. 1, n. 2, p. 146--160, 1972.
- AHO, A. V.; HOPCROFT, J. E.; ULLMAN, J. D. *Data Structures and Algorithms*.
  Addison-Wesley, 1983. Seção 6.7: o algoritmo de Kosaraju (1978, inédito).
- SHARIR, M. A strong-connectivity algorithm and its applications in data flow
  analysis. *Comput. Math. Appl.*, v. 7, n. 1, p. 67--72, 1981.

Lista completa em `../referencias.bib`.
